# Image Quality Degradation Validation: Noise, Contrast, Blur & Smooth

This notebook evaluates the behavior of pose estimation under systematic image quality corruptions.

Key configuration controls in `config.yaml`:
- `dataset.noise_sigma`: Additive white Gaussian noise
- `dataset.contrast_factor`: Multiplicative contrast scaling
- `dataset.blur_kernel_size`: Gaussian spatial blur
- `dataset.smooth_kernel_size`: Low-pass uniform averaging box filter

In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root dynamically
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd().parent
)
PROJECT_ROOT = project_root
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
mmpose_root = project_root / "models" / "mmpose"
if mmpose_root.exists() and str(mmpose_root) not in sys.path:
    sys.path.insert(0, str(mmpose_root))

print(f"Project root: {project_root}")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

from pose_uncertainty.data_loader import COCOLoader
from pose_uncertainty.utils.types import ImageSample

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams['figure.dpi'] = 120

print("Notebook ready for input image quality degradation analysis.")

# Reference Sample Loading

Loads a standardized benchmark sample to compare identical image content under varying corruption parameters.

In [ ]:
dataset_root = project_root / "data" / "coco"
annotation_file = "annotations/person_keypoints_val2017.json"
image_subdir = "val2017"
idx = 10

base_loader_args = {
    "data_root": str(dataset_root),
    "ann_file": annotation_file,
    "image_dir": image_subdir,
    "resize_scale": 1.0,
    "noise_mean": 0.0,
    "noise_sigma": 0.0,
    "noise_seed": 42,
    "contrast_factor": 1.0,
    "blur_kernel_size": 0,
    "blur_sigma": 0.0,
    "smooth_kernel_size": 0,
}

def get_sample(extra_kwargs=None):
    loader_kwargs = dict(base_loader_args)
    if extra_kwargs:
        loader_kwargs.update(extra_kwargs)
    loader = COCOLoader(**loader_kwargs)
    return next(itertools.islice(loader, idx, idx + 1))

sample_ref = get_sample()
print(f"Referencia: image_id={sample_ref.image_id}, shape={sample_ref.image_array.shape}, bbox={tuple(round(v, 2) for v in sample_ref.bbox)}")

In [ ]:
def draw_sample(ax, sample: ImageSample, title: str) -> None:
    ax.imshow(sample.image_array)
    ax.set_title(title)
    ax.axis("off")

    x, y, w, h = sample.bbox
    ax.add_patch(
        plt.Rectangle(
            (x, y),
            w,
            h,
            fill=False,
            edgecolor="cyan",
            linewidth=2,
        )
    )

    for kp in sample.ground_truth_keypoints or []:
        ax.scatter(kp.x, kp.y, s=16, c="yellow", edgecolors="black", linewidths=0.4)
        ax.text(kp.x + 1.5, kp.y + 1.5, kp.name, fontsize=6, color="white")

def compare_effect(effect_name: str, variant_specs: list[tuple[str, dict]]) -> pd.DataFrame:
    samples = {"base": sample_ref}
    for label, kwargs in variant_specs:
        samples[label] = get_sample(kwargs)

    rows = []
    for label, sample in samples.items():
        rows.append({
            "variant": label,
            "image_width": sample.image_array.shape[1],
            "image_height": sample.image_array.shape[0],
            "bbox_x": round(sample.bbox[0], 2),
            "bbox_y": round(sample.bbox[1], 2),
            "bbox_w": round(sample.bbox[2], 2),
            "bbox_h": round(sample.bbox[3], 2),
        })

    df = pd.DataFrame(rows)
    display(df)

    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 6))
    if len(samples) == 1:
        axes = [axes]

    for ax, (label, sample) in zip(axes, samples.items()):
        draw_sample(ax, sample, f"{effect_name} | {label}")

    plt.tight_layout()
    plt.show()
    return df

# Visual Comparison Across Corruption Types

Compares baseline vs corrupted inputs to inspect signal degradation and model failure modes.

In [ ]:
noise_df = compare_effect(
    "Ruido blanco gaussiano",
    [
        ("sigma=100", {"noise_sigma": 100.0}),
        ("sigma=200", {"noise_sigma": 200.0}),
        ("sigma=300", {"noise_sigma": 300.0}),
        ("sigma=400", {"noise_sigma": 400.0}),
    ],
)

contrast_df = compare_effect(
    "Contraste",
    [
        ("factor=0.75", {"contrast_factor": 0.75}),
        ("factor=0.50", {"contrast_factor": 0.50}),
        ("factor=0.25", {"contrast_factor": 0.25}),
        ("factor=0.10", {"contrast_factor": 0.10}),
        ("factor=0.05", {"contrast_factor": 0.05}),
    ],
)

blur_df = compare_effect(
    "Blur",
    [
        ("k=21", {"blur_kernel_size": 21}),
        ("k=31", {"blur_kernel_size": 31}),
        ("k=41", {"blur_kernel_size": 41}),
        ("k=51", {"blur_kernel_size": 51}),
        ("k=61", {"blur_kernel_size": 61}),
        ("k=71", {"blur_kernel_size": 71}),
    ],
)

smooth_df = compare_effect(
    "Smooth / low-pass",
    [
        ("k=10", {"smooth_kernel_size": 10}),
        ("k=20", {"smooth_kernel_size": 20}),
        ("k=30", {"smooth_kernel_size": 30}),
        ("k=40", {"smooth_kernel_size": 40}),
    ],
)

In [ ]:
output_dir = project_root / "outputs" / "notebooks" / "image_quality_validation"
output_dir.mkdir(parents=True, exist_ok=True)

for name, df in {
    "noise": noise_df,
    "contrast": contrast_df,
    "blur": blur_df,
    "smooth": smooth_df,
}.items():
    df.to_csv(output_dir / f"{name}_comparison.csv", index=False)

print(f"Saved metric comparisons to: {output_dir}")

benchmark_commands = [
    "python src/experiments/benchmarks/run_benchmark.py dataset.noise_sigma=15 dataset.noise_seed=42 force_rerun=true",
    "python src/experiments/benchmarks/run_benchmark.py dataset.contrast_factor=0.6 force_rerun=true",
    "python src/experiments/benchmarks/run_benchmark.py dataset.blur_kernel_size=7 force_rerun=true",
    "python src/experiments/benchmarks/run_benchmark.py dataset.smooth_kernel_size=7 force_rerun=true",
]

print("Recommended CLI Benchmark Commands:")
for command in benchmark_commands:
    print("  ", command)

# Key Findings & Benchmark Execution Guidelines

- Parameter overrides can be passed directly via CLI to `run_benchmark.py`.
- Gaussian blur and box filter smoothing exhibit distinct spatial frequency cutoffs, providing comprehensive stress-testing for heatmap sharpness weighting.